# 🏷️ Chapter 5: Classification
**Reference:** *Practical Statistics for Data Scientists (2nd Edition)* by Peter Bruce, Andrew Bruce, & Peter Gedeck

---

## 5.1 Introduction to Classification
In Chapter 4, we used Regression to predict a continuous numerical value (e.g., House Price). In this chapter, we use **Classification** to predict a categorical outcome (e.g., Will this borrower Default or Pay Off their loan? Is this email Spam or Not Spam?).

While machine learning offers complex black-box classifiers (like Random Forests and Neural Networks), classical statistics provides highly interpretable, mathematically grounded classification models. We will cover the big three: **Naive Bayes, Discriminant Analysis, and Logistic Regression**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.utils import resample

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')
np.random.seed(42)

print("Libraries successfully imported for Chapter 5.")

## 5.2 Creating a Synthetic Loan Default Dataset
To demonstrate these algorithms, we will create a synthetic dataset representing bank loans. The target variable is `Default` (1 if they defaulted, 0 if they paid it off). The predictors will be `Credit_Score`, `Income`, and `Debt_to_Income`.

In [ ]:
# Generate Synthetic Loan Data
n_samples = 1000

# Predictors
credit_score = np.random.normal(650, 80, n_samples)
income = np.random.normal(60000, 20000, n_samples)
debt_to_income = np.random.uniform(0.1, 0.6, n_samples)

# Calculate a hidden 'risk score' to determine default probability
# Lower credit score, lower income, and higher DTI increase risk
risk_score = -0.01 * credit_score - 0.00005 * income + 5 * debt_to_income + 7

# Sigmoid function to convert risk score to a probability [0, 1]
probabilities = 1 / (1 + np.exp(-risk_score))

# Generate target variable based on probabilities (1 = Default, 0 = Paid)
default = np.random.binomial(1, probabilities)

loan_df = pd.DataFrame({
    'Credit_Score': credit_score,
    'Income': income,
    'DTI': debt_to_income,
    'Default': default
})

# Split into Train and Test sets (80% train, 20% test)
X = loan_df[['Credit_Score', 'Income', 'DTI']]
y = loan_df['Default']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Dataset Sample:")
print(loan_df.head())
print(f"\nTotal Defaults: {sum(default)} out of {n_samples} ({(sum(default)/n_samples)*100:.1f}%)")

## 5.3 Naive Bayes
The Naive Bayes algorithm uses probability to classify records. It is based on **Bayes' Rule**, which updates our belief about a record belonging to a specific class given its predictor values.

Why is it called "Naive"? 
Because the algorithm mathematically assumes that **all predictor variables are completely independent of each other**. In the real world, Income and Credit Score are usually correlated, so this assumption is strictly false. However, despite being "naive," the algorithm works surprisingly well in practice, especially for text classification (like spam filters).

In [ ]:
# Initialize and train the Gaussian Naive Bayes model
# (Gaussian assumes the continuous features follow a normal distribution)
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Predict on the test set
nb_predictions = nb_model.predict(X_test)

print("Naive Bayes Accuracy:", np.round(accuracy_score(y_test, nb_predictions), 3))

## 5.4 Discriminant Analysis (LDA)
Linear Discriminant Analysis (LDA) is the oldest classification technique, developed by R.A. Fisher in 1936.

Instead of looking at probabilities, LDA looks at **Covariance** (how variables move together). It attempts to find a linear combination of features (a line or plane) that maximizes the distance *between* the means of the two classes, while minimizing the variance *within* each class.

Visually, LDA tries to draw an axis through the data where the "Default" clump of data and the "Paid" clump of data are pushed as far apart as possible.

In [ ]:
# Initialize and train the LDA model
lda_model = LinearDiscriminantAnalysis()
lda_model.fit(X_train, y_train)

# Predict on the test set
lda_predictions = lda_model.predict(X_test)

print("LDA Accuracy:", np.round(accuracy_score(y_test, lda_predictions), 3))

# LDA provides interpretable coefficients (which features push the score towards Default)
lda_coefs = pd.DataFrame(lda_model.coef_[0], index=X.columns, columns=['LDA Weight'])
print("\nLDA Feature Weights:")
print(lda_coefs)

## 5.5 Logistic Regression
Logistic Regression is the gold standard of statistical classification. Despite the name "regression," it is a classification algorithm.

**The Problem:** We want to predict a probability between 0 and 1. If we use standard linear regression, our model will eventually draw a line that predicts probabilities like `-0.5` or `1.2`, which is mathematically impossible.

**The Solution:** We apply the **Logistic (Sigmoid) Function**. Instead of modeling the probability directly, we model the **Log-Odds** (the logarithm of the odds ratio). The log-odds can map any number from $-\infty$ to $+\infty$, making it perfect for a linear equation.

$$ p = \frac{1}{1 + e^{-(\beta_0 + \beta_1X_1 + \dots + \beta_nX_n)}} $$

Because it outputs a strict probability, Logistic Regression gives us incredible confidence in its predictions, and its coefficients can be interpreted as odds multipliers.

In [ ]:
# Initialize and train the Logistic Regression model
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

log_predictions = log_reg.predict(X_test)

print("Logistic Regression Accuracy:", np.round(accuracy_score(y_test, log_predictions), 3))

## 5.6 Evaluating Classification Models
In regression, we used RMSE to evaluate error. In classification, **Accuracy** (Percentage of correct guesses) is often a terrible metric. If only 1% of transactions are fraudulent, a model that simply guesses "Not Fraud" every single time will be 99% accurate, but completely useless!

We need better metrics derived from the **Confusion Matrix**.

### The Confusion Matrix
A 2x2 table that categorizes predictions into four buckets:
- **True Positives (TP):** Model predicted Default, and they Defaulted.
- **True Negatives (TN):** Model predicted Paid, and they Paid.
- **False Positives (FP):** Model predicted Default, but they Paid (Type I Error).
- **False Negatives (FN):** Model predicted Paid, but they Defaulted (Type II Error - Often the most expensive error!).

### Key Metrics
- **Precision:** Out of all the people we *predicted* would default, how many actually did? (TP / (TP + FP))
- **Recall (Sensitivity):** Out of all the people who *actually* defaulted, how many did we catch? (TP / (TP + FN))
- **Specificity:** Out of all the people who *actually* paid, how many did we correctly identify? (TN / (TN + FP))

In [ ]:
# Generate the Confusion Matrix for Logistic Regression
conf_matrix = confusion_matrix(y_test, log_predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted Paid', 'Predicted Default'],
            yticklabels=['Actual Paid', 'Actual Default'])
plt.title('Confusion Matrix: Logistic Regression')
plt.show()

# Print the Detailed Classification Report
print("Classification Report:\n")
print(classification_report(y_test, log_predictions, target_names=['Paid (0)', 'Default (1)']))

## 5.7 The ROC Curve and AUC
When a model predicts a class, it actually generates a probability (e.g., "80% chance of Default"). By default, if the probability is > 0.5, it classifies it as a Default. However, we can change this threshold to be more conservative or aggressive.

The **ROC Curve (Receiver Operating Characteristic)** plots the Recall (True Positive Rate) against the False Positive Rate across *every possible threshold* (from 0.0 to 1.0).

The **AUC (Area Under the Curve)** gives a single number to grade the model. 
- AUC = 0.5 means the model is just randomly guessing (a diagonal line).
- AUC = 1.0 means the model is perfectly separating the classes.

In [ ]:
# Get predicted probabilities (not just the 0/1 classes) for the Default class (column 1)
log_probs = log_reg.predict_proba(X_test)[:, 1]

# Calculate False Positive Rate (FPR) and True Positive Rate (TPR) across thresholds
fpr, tpr, thresholds = roc_curve(y_test, log_probs)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(7, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--', label='Random Guessing (AUC = 0.50)')

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Recall)')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc="lower right")
plt.grid(alpha=0.3)
plt.show()

## 5.8 Strategies for Imbalanced Data
If you are predicting ad clicks, fraud, or rare diseases, your target variable might only represent 1% of the data. The model will heavily bias itself toward the majority class.

To fix this, we manipulate the training data before feeding it to the model:
1. **Undersampling:** Throw away a large portion of the majority class (e.g., non-fraud) until the classes are balanced (50/50). Fast, but you lose data.
2. **Oversampling:** Duplicate the rare class records until the classes are balanced.
3. **SMOTE (Synthetic Minority Oversampling Technique):** Instead of duplicating rare records, SMOTE mathematically generates brand new, synthetic records that are statistically similar to the rare records by interpolating between them.

In [ ]:
# Let's demonstrate manual Oversampling
# Separate the training data into the majority (Paid) and minority (Default) classes
X_train_full = X_train.copy()
X_train_full['Default'] = y_train

majority_class = X_train_full[X_train_full['Default'] == 0]
minority_class = X_train_full[X_train_full['Default'] == 1]

print(f"Original Imbalance - Paid: {len(majority_class)}, Default: {len(minority_class)}")

# Upsample the minority class to match the majority class size
minority_upsampled = resample(minority_class, 
                              replace=True,     # Sample with replacement
                              n_samples=len(majority_class),    # Match majority size
                              random_state=42) 

# Combine majority class with the newly upsampled minority class
upsampled_train_data = pd.concat([majority_class, minority_upsampled])

print(f"New Balanced Data  - Paid: {len(upsampled_train_data[upsampled_train_data['Default'] == 0])}, Default: {len(upsampled_train_data[upsampled_train_data['Default'] == 1])}")

# If we retrain the model on this balanced data, its overall Accuracy might drop, 
# but its Recall (ability to catch defaults) will drastically improve!

### Conclusion of Chapter 5
Classification allows us to categorize records based on data. 

**Key Takeaways:**
1. **Logistic Regression** is the foundation of statistical classification because it natively models probabilities using the log-odds transformation.
2. **Accuracy is a trap.** Always evaluate classification models using the Confusion Matrix, checking Precision and Recall based on the business context (e.g., is a False Positive or False Negative more expensive?).
3. **AUC (Area Under the Curve)** is the best single-number metric to compare different models across all probability thresholds.
4. When dealing with rare events, you must use **Oversampling, Undersampling, or SMOTE** to force the model to pay attention to the minority class.